In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get("RIDESHARE_RAW_DIR", str(PROJECT_DIR / "data/raw/fhvhv")))
PROCESSED_DIR = Path(
    os.environ.get("RIDESHARE_PROCESSED_DIR", str(PROJECT_DIR / "data/processed"))
)
ZONE_DIR = Path(
    os.environ.get("RIDESHARE_ZONE_DIR", str(PROJECT_DIR / "data/reference/taxi_zones"))
)
WEATHER_DIR = Path(
    os.environ.get(
        "RIDESHARE_WEATHER_DIR", str(PROJECT_DIR / "data/reference/weather_hourly")
    )
)
ANALYSIS_DIR = PROJECT_DIR / "outputs/demand_analysis"
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.


# All external paths and run choices are here. Use a fresh directory for a new run.
OUTPUT_DIR = PROCESSED_DIR
SOURCE_YEARS = [2019, 2021, 2022, 2023, 2024, 2025]
POLICY_DATE = "2025-01-05"
WRITE_MODE = "errorifexists"  # Never overwrite the original research outputs.


# 02 Prepare hourly demand aggregates from 2019 to 2025

## Purpose and run order
This is the single sequential preparation notebook for **February–December 2019 and all months of 2021–2025**, excluding 2020. It replaces the separate preparation and selected-repair notebooks for a fresh reproduction. Run all cells once, in order, assuming the required source files and a fresh output directory are available.

For each month the workflow loads raw records, cleans trips, derives features, creates pickup/dropoff/OD hourly aggregates, reads them back, and records validation results. It stops at the first failure. There are no restart, repair, checkpoint-resume or automatic replacement branches. Notebook 02 is deliberately published **without execution outputs**; the existing research datasets were not regenerated for this documentation revision.

All recorded HVFHV platforms are included. Only the first configuration cell contains directory choices. Java and Hadoop configuration belong in the environment. Downstream analysis should point to the same processed directory; notebooks 03–05 select 2021–2025.


## 1 Validate the source manifest
HVFHV reporting starts in February 2019. Require every selected source month before starting Spark or creating analytical outputs.


In [ ]:
import calendar
import json
import pandas as pd
from IPython.display import display

TARGET_MONTHS = [
    (year, month)
    for year in SOURCE_YEARS
    for month in range(1, 13)
    if (year, month) != (2019, 1)
]
assert len(TARGET_MONTHS) == 71 and all(year != 2020 for year, _ in TARGET_MONTHS)
sources = {
    (year, month): RAW_DIR / f"fhvhv_tripdata_{year}-{month:02d}.parquet"
    for year, month in TARGET_MONTHS
}
assert all(
    path.is_file() for path in sources.values()
), "Complete the 71-month source inventory first."
# Fail before expensive work if any selected destination already exists.
stems = {
    "pickup": "fhvhv_by_zone_hour",
    "dropoff": "fhvhv_by_DOzone_hour",
    "od": "fhvhv_by_OD_hour",
}
destinations = [
    OUTPUT_DIR / f"{stem}_{year}_{month:02d}.parquet"
    for year, month in TARGET_MONTHS
    for stem in stems.values()
]
assert not any(
    p.exists() for p in destinations
), "Select a fresh OUTPUT_DIR; existing results are preserved."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(
    pd.DataFrame(TARGET_MONTHS, columns=["year", "month"])
    .groupby("year")
    .size()
    .rename("source_months")
)


## 2 Start Spark
One Spark session processes the months sequentially. Disk persistence prevents repeatedly recomputing the cleaned month while writing three aggregate grains. This stage can require substantial disk space and runtime.


In [ ]:
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("TLC_Hourly_Preparation_2019_2025")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## 3 Clean one source month and derive trip features
Remove exact duplicates, require positive distance/duration/fare/pay, reject invalid request order, and retain distance ≤200 miles, calculated duration ≤10,800 seconds and speed in (1,120] mph. Missing CBD fees are filled as in the established workflow. Fare rates will be calculated after aggregation, not by averaging row-level ratios.


In [ ]:
def prepare_month(source, year):
    # Only one monthly file is scanned and cleaned at a time.
    df = spark.read.parquet(str(source))
    if year <= 2023:
        df = df.withColumn("cbd_congestion_fee", F.lit(0.0))
        if "airport_fee" not in df.columns:
            df = df.withColumn("airport_fee", F.lit(0.0))
        df = df.withColumn("airport_fee", F.coalesce(F.col("airport_fee"), F.lit(0.0)))
    if "cbd_congestion_fee" not in df.columns:
        # Earlier schemas lack this field; add null before the original cleaning.
        df = df.withColumn("cbd_congestion_fee", F.lit(None).cast("double"))
    df = df.withColumn(
        "trip_time_cal",
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"),
    ).withColumn("trip_speed", (F.col("trip_miles") / F.col("trip_time")) * 3600)
    # Exact-row deduplication precedes fee coalescing and the selected filters.
    # Speed uses reported trip_time; duration uses timestamp subtraction.
    df_clean = (
        df.dropDuplicates()
        .filter(F.col("trip_miles") > 0)
        .filter(F.col("trip_time_cal") > 0)
        .filter(F.col("base_passenger_fare") > 0)
        .filter(F.col("trip_speed") > 1)
        .filter(F.col("driver_pay") > 0)
        .filter(F.col("pickup_datetime") >= F.col("request_datetime"))
        .filter(F.col("trip_miles") <= 200)
        .filter(F.col("trip_time_cal") <= 10800)
        .filter(F.col("trip_speed") <= 120)
        .withColumn(
            "cbd_congestion_fee", F.coalesce(F.col("cbd_congestion_fee"), F.lit(0.0))
        )
    )
    return (
        df_clean.withColumn("year", F.year("pickup_datetime"))
        .withColumn("month", F.month("pickup_datetime"))
        .withColumn("week", F.weekofyear("pickup_datetime"))
        .withColumn("date", F.to_date("pickup_datetime"))
        .withColumn("hour", F.hour("pickup_datetime"))
        .withColumn("do_hour", F.hour("dropoff_datetime"))
        .withColumn("day_of_week", F.dayofweek("pickup_datetime"))
        .withColumn(
            "is_weekend", F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)
        )
        .withColumn(
            "policy_period",
            F.when(
                F.col("pickup_datetime") < F.lit(POLICY_DATE), "pre_policy"
            ).otherwise("post_policy"),
        )
        .withColumn(
            "wait_time_sec",
            F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime"),
        )
        .withColumn("trip_duration_min", F.col("trip_time_cal") / 60)
        .withColumn(
            "total_fare",
            F.col("base_passenger_fare")
            + F.col("tolls")
            + F.col("tips")
            + F.col("congestion_surcharge")
            + F.col("sales_tax")
            + F.col("bcf")
            + F.col("airport_fee")
            + F.col("cbd_congestion_fee"),
        )
        .withColumn(
            "tnc_comp",
            F.col("base_passenger_fare") + F.col("tips") - F.col("driver_pay"),
        )
    )


## 4 Define aggregate metrics
Pickup/dropoff outputs contain trip counts and aggregate-first financial/operational rates. OD outputs retain the established count and average-fare/distance/duration fields. Keeping formulas in named functions makes the processing loop easier to follow.


In [ ]:
def compute_metrics(df, group_cols):
    """
    Aggregate all metrics using aggregate-first approach.
    Group by group_cols and compute all metrics.
    """
    return (
        df.groupBy(group_cols)
        .agg(
            # Demand
            F.count("*").alias("trip_count"),
            # Intermediate sums for rate calculations
            F.sum("wait_time_sec").alias("_sum_wait_time"),
            F.sum("base_passenger_fare").alias("_sum_base_fare"),
            F.sum("total_fare").alias("_sum_total_fare"),
            F.sum("driver_pay").alias("_sum_driver_pay"),
            F.sum("tnc_comp").alias("_sum_tnc_comp"),
            F.sum("trip_miles").alias("_sum_trip_miles"),
            F.sum("trip_duration_min").alias("_sum_trip_duration_min"),
        )
        .withColumn(
            "avg_wait_time_sec",
            F.round(F.col("_sum_wait_time") / F.col("trip_count"), 2),
        )
        .withColumn(
            "base_fare_per_mile",
            F.round(F.col("_sum_base_fare") / F.col("_sum_trip_miles"), 4),
        )
        .withColumn(
            "base_fare_per_min",
            F.round(F.col("_sum_base_fare") / F.col("_sum_trip_duration_min"), 4),
        )
        .withColumn(
            "base_fare_per_trip",
            F.round(F.col("_sum_base_fare") / F.col("trip_count"), 4),
        )
        .withColumn(
            "total_fare_per_trip",
            F.round(F.col("_sum_total_fare") / F.col("trip_count"), 4),
        )
        .withColumn(
            "driver_pay_per_mile",
            F.round(F.col("_sum_driver_pay") / F.col("_sum_trip_miles"), 4),
        )
        .withColumn(
            "driver_pay_per_min",
            F.round(F.col("_sum_driver_pay") / F.col("_sum_trip_duration_min"), 4),
        )
        .withColumn(
            "driver_pay_per_trip",
            F.round(F.col("_sum_driver_pay") / F.col("trip_count"), 4),
        )
        .withColumn(
            "tnc_commission_rate",
            F.round(F.col("_sum_tnc_comp") / F.col("_sum_base_fare"), 4),
        )
        .drop(
            "_sum_wait_time",
            "_sum_base_fare",
            "_sum_total_fare",
            "_sum_driver_pay",
            "_sum_tnc_comp",
            "_sum_trip_miles",
            "_sum_trip_duration_min",
        )
    )


In [ ]:
def aggregate_od(df):
    keys = [
        "PULocationID",
        "DOLocationID",
        "date",
        "year",
        "month",
        "week",
        "day_of_week",
        "is_weekend",
        "hour",
        "policy_period",
    ]
    return (
        df.groupBy(keys)
        .agg(
            F.count("*").alias("trip_count"),
            F.sum("base_passenger_fare").alias("_sum_base_fare"),
            F.sum("total_fare").alias("_sum_total_fare"),
            F.sum("trip_miles").alias("_sum_trip_miles"),
            F.sum("trip_duration_min").alias("_sum_trip_duration_min"),
            F.sum("cbd_congestion_fee").alias("total_cbd_fee"),
        )
        .withColumn(
            "avg_base_fare", F.round(F.col("_sum_base_fare") / F.col("trip_count"), 4)
        )
        .withColumn(
            "avg_total_fare", F.round(F.col("_sum_total_fare") / F.col("trip_count"), 4)
        )
        .withColumn(
            "avg_trip_miles", F.round(F.col("_sum_trip_miles") / F.col("trip_count"), 4)
        )
        .withColumn(
            "avg_duration_min",
            F.round(F.col("_sum_trip_duration_min") / F.col("trip_count"), 4),
        )
        .drop(
            "_sum_base_fare",
            "_sum_total_fare",
            "_sum_trip_miles",
            "_sum_trip_duration_min",
        )
    )


## 5 Validate and write each hourly table
Validation checks output readability, positive counts, unique zone/date/hour keys, month/date coverage and count reconciliation against the selected cleaned rows. A successful Spark write alone is not enough.


In [ ]:
def calendar_columns(frame):
    # Recalculate the calendar descriptors from the aggregate's date key.
    return (
        frame.withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
        .withColumn("week", F.weekofyear("date"))
        .withColumn("day_of_week", F.dayofweek("date"))
        .withColumn(
            "is_weekend", F.when(F.dayofweek("date").isin(1, 7), 1).otherwise(0)
        )
        .withColumn(
            "policy_period",
            F.when(F.col("date") < F.lit(POLICY_DATE), "pre_policy").otherwise(
                "post_policy"
            ),
        )
    )


def write_and_validate(frame, selected, family, year, month):
    folder = OUTPUT_DIR / f"{stems[family]}_{year}_{month:02d}.parquet"
    frame.write.mode(WRITE_MODE).parquet(str(folder))
    observed = spark.read.parquet(str(folder))
    zone_keys = {
        "pickup": ["PULocationID"],
        "dropoff": ["DOLocationID"],
        "od": ["PULocationID", "DOLocationID"],
    }[family]
    hour_key = "do_hour" if family == "dropoff" else "hour"
    keys = zone_keys + ["date", hour_key]
    invalid = observed.filter(
        F.col("trip_count").isNull()
        | (F.col("trip_count") <= 0)
        | (F.col(hour_key) < 0)
        | (F.col(hour_key) > 23)
    )
    assert invalid.count() == 0, f"Invalid counts/hours: {folder.name}"
    assert observed.select(keys).dropna().count() == observed.count(), "Null output key"
    assert (
        observed.groupBy(keys).count().filter(F.col("count") > 1).count() == 0
    ), "Duplicate key"
    assert (
        observed.select("date").distinct().count()
        == calendar.monthrange(year, month)[1]
    )
    assert (
        observed.filter((F.year("date") != year) | (F.month("date") != month)).count()
        == 0
    )
    cleaned_rows = selected.count()
    aggregate_trips = int(observed.agg(F.sum("trip_count")).first()[0])
    assert aggregate_trips == cleaned_rows, "Raw-to-aggregate count mismatch"
    assert (folder / "_SUCCESS").exists() and list(folder.glob("*.parquet"))
    result = dict(
        version="hourly-preparation-v1",
        family=family,
        year=year,
        month=month,
        cleaned_rows=cleaned_rows,
        aggregate_trips=aggregate_trips,
        aggregate_rows=observed.count(),
        passed=True,
    )
    (folder / "_preparation_validation.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    return result


## 6 Process all 71 months in one sequential run
The current month is materialized once and reused for pickup and OD. At most one preceding cleaned month is retained when needed by the next monthly computation. Data never come from 2020. There is no reuse of partial output directories; a fresh run creates all 213 tables.


In [ ]:
summary = []
previous, previous_key = None, None
calendar_keys = ["date", "year", "month", "week", "day_of_week", "is_weekend"]
try:
    for year, month in TARGET_MONTHS:
        print(f"Preparing {year}-{month:02d}")
        current = prepare_month(sources[year, month], year).persist(
            StorageLevel.DISK_ONLY
        )
        # Verify the provider file belongs to its named pickup month.
        assert (
            current.filter(
                (F.year("pickup_datetime") != year)
                | (F.month("pickup_datetime") != month)
            ).count()
            == 0
        )
        assert current.count() > 0, "Cleaning produced an empty month"

        pickup_keys = ["PULocationID"] + calendar_keys + ["hour", "policy_period"]
        pickup = compute_metrics(current, pickup_keys)
        summary.append(write_and_validate(pickup, current, "pickup", year, month))
        summary.append(
            write_and_validate(aggregate_od(current), current, "od", year, month)
        )

        drop_source = current
        if year <= 2023:
            prior = (year - 1, 12) if month == 1 else (year, month - 1)
            if previous is not None and previous_key == prior:
                drop_source = previous.unionByName(current, allowMissingColumns=True)
            drop_source = drop_source.withColumn("date", F.to_date("dropoff_datetime"))
            drop_source = drop_source.filter(
                (F.year("date") == year) & (F.month("date") == month)
            )
        drop_source = calendar_columns(drop_source)
        drop_keys = ["DOLocationID"] + calendar_keys + ["do_hour", "policy_period"]
        dropoff = compute_metrics(drop_source, drop_keys)
        summary.append(write_and_validate(dropoff, drop_source, "dropoff", year, month))

        # Release the preceding month only after the current outputs are validated.
        if previous is not None:
            previous.unpersist()
        if year <= 2023:
            previous, previous_key = current, (year, month)
        else:
            current.unpersist()
            previous, previous_key = None, None
        spark.catalog.clearCache() if previous is None else None

    validation = pd.DataFrame(summary)
    assert len(validation) == 213 and validation.passed.all()
    validation.to_csv(OUTPUT_DIR / "preparation_validation_summary.csv", index=False)
    display(
        validation.groupby(["year", "family"]).agg(
            months=("month", "count"), trips=("aggregate_trips", "sum")
        )
    )
finally:
    if previous is not None:
        previous.unpersist()
    spark.stop()


## Completed preparation and next steps
On a successful fresh run the summary contains 213 validated monthly tables: 71 pickup, 71 dropoff and 71 OD. Point notebooks 03–06 to this output directory. Notebook 03–05 plots remain restricted to 2021–2025; 2019 is prepared for the broader research archive. January 2026 held-out pickup targets are prepared separately using the documented evaluation script. This notebook is not executed as part of the documentation refresh.
